# Agent pipeline — testbench

Notebook to manually test the building blocks of the LangGraph agent (data loading, nodes, models) in isolation, before wiring them into the full graph.

> Select the project's `venv` as the kernel (`venv/bin/python`) before running.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

## Infrastructure and application state

Loads the running example data (nodes, links, services) used across the notebook.

In [2]:
from agent.utils.config import load_config
from agent.utils.data_loader import load_links, load_nodes, load_services

config = load_config()
nodes = load_nodes(config.data.nodes_csv)
links = load_links(config.data.links_csv)
services = load_services(config.data.app_state_json)

print(f"{len(nodes)} nodes, {len(links)} links, {len(services)} services")
nodes

6 nodes, 9 links, 5 services


{'N1': Node(node_id='N1', node_name='Pepper', tier='iot', node_type='robot', cpu_cores=4, cpu_freq_ghz=1.8, ram_gb=8, storage_gb=32, mobility='mobile'),
 'N2': Node(node_id='N2', node_name='RaspberryPi4', tier='edge', node_type='embedded_edge', cpu_cores=4, cpu_freq_ghz=1.5, ram_gb=4, storage_gb=32, mobility='static'),
 'N3': Node(node_id='N3', node_name='LaptopEdge', tier='edge', node_type='laptop', cpu_cores=8, cpu_freq_ghz=3.2, ram_gb=16, storage_gb=512, mobility='mobile'),
 'N4': Node(node_id='N4', node_name='JetsonNano', tier='edge', node_type='edge_ai', cpu_cores=6, cpu_freq_ghz=1.4, ram_gb=8, storage_gb=128, mobility='static'),
 'N5': Node(node_id='N5', node_name='IntelNUC_Fog', tier='fog', node_type='fog_node', cpu_cores=8, cpu_freq_ghz=3.6, ram_gb=16, storage_gb=512, mobility='static'),
 'N7': Node(node_id='N7', node_name='Cloud_A100', tier='cloud', node_type='cloud_gpu_node', cpu_cores=96, cpu_freq_ghz=2.5, ram_gb=900, storage_gb=10000, mobility='static')}

In [3]:
services

{'T1': Service(service_id='T1', name='Capture', description='Video stream acquisition', current_node='N1', requirements=None),
 'T2': Service(service_id='T2', name='Preprocessing', description='Resizing', current_node='N1', requirements=None),
 'T3': Service(service_id='T3', name='Detection', description='Object detection', current_node='N5', requirements={'cpu_cores': 4.0, 'ram_gb': 8.0}),
 'T4': Service(service_id='T4', name='Tracking', description='Multi-frame object tracking', current_node='N5', requirements=None),
 'T5': Service(service_id='T5', name='Storage/Big Data Analysis', description='Archiving', current_node='N7', requirements=None)}

## Intent Grounding node

Builds an `AgentState` for the running example intent (scenario 1: detection model update) and runs `intent_grounding_node` on it.

In [4]:
from agent.states.state import AgentState

INTENT = (
    "The detection model has been updated to a more accurate version, "
    "it now requires at least 12 vCPU and 24GB of RAM to run correctly."
)

state: AgentState = {
    "intent_text": INTENT,
    "services": services,
    "nodes": nodes,
    "links": links,
    "requirements": [],
    "necessity_result": None,
    "decision_result": None,
    "explanation": None,
}
state

{'intent_text': 'The detection model has been updated to a more accurate version, it now requires at least 12 vCPU and 24GB of RAM to run correctly.',
 'services': {'T1': Service(service_id='T1', name='Capture', description='Video stream acquisition', current_node='N1', requirements=None),
  'T2': Service(service_id='T2', name='Preprocessing', description='Resizing', current_node='N1', requirements=None),
  'T3': Service(service_id='T3', name='Detection', description='Object detection', current_node='N5', requirements={'cpu_cores': 4.0, 'ram_gb': 8.0}),
  'T4': Service(service_id='T4', name='Tracking', description='Multi-frame object tracking', current_node='N5', requirements=None),
  'T5': Service(service_id='T5', name='Storage/Big Data Analysis', description='Archiving', current_node='N7', requirements=None)},
 'nodes': {'N1': Node(node_id='N1', node_name='Pepper', tier='iot', node_type='robot', cpu_cores=4, cpu_freq_ghz=1.8, ram_gb=8, storage_gb=32, mobility='mobile'),
  'N2': Node(

In [5]:
from agent.prompts.requirement_extraction import build_user_prompt

user_prompt = build_user_prompt(state["intent_text"], state["services"])
print(user_prompt)

Application services:
- T1: Capture (Video stream acquisition)
- T2: Preprocessing (Resizing)
- T3: Detection (Object detection)
- T4: Tracking (Multi-frame object tracking)
- T5: Storage/Big Data Analysis (Archiving)

User intent:
"The detection model has been updated to a more accurate version, it now requires at least 12 vCPU and 24GB of RAM to run correctly."


In [6]:
from agent.utils.llm import get_mistral_llm, get_openai_llm, get_meta_llm, get_nvidia_llm

for name, get_llm in [("mistral", get_mistral_llm), ("openai", get_openai_llm), ("meta", get_meta_llm), ("nvidia", get_nvidia_llm)]:
    llm = get_llm()
    resolved_key = llm._client.api_key.get_secret_value()
    print(f"[{name}]")
    print(f"  type       : {type(llm).__name__}")
    print(f"  model      : {llm.model}")
    print(f"  base_url   : {llm.base_url}")
    print(f"  temperature: {llm.temperature}")
    print(f"  max_tokens : {llm.max_tokens}")
    print(f"  api_key set: {bool(resolved_key)} (length={len(resolved_key)})")
    print()

[mistral]
  type       : ChatNVIDIA
  model      : mistralai/mistral-nemotron
  base_url   : https://integrate.api.nvidia.com/v1
  temperature: 0.0
  max_tokens : 4096
  api_key set: True (length=70)

[openai]
  type       : ChatNVIDIA
  model      : openai/gpt-oss-20b
  base_url   : https://integrate.api.nvidia.com/v1
  temperature: 0.0
  max_tokens : 4096
  api_key set: True (length=70)

[meta]
  type       : ChatNVIDIA
  model      : meta/llama-3.2-11b-vision-instruct
  base_url   : https://integrate.api.nvidia.com/v1
  temperature: 0.0
  max_tokens : 4096
  api_key set: True (length=70)

[nvidia]
  type       : ChatNVIDIA
  model      : nvidia/nemotron-3-nano-omni-30b-a3b-reasoning
  base_url   : https://integrate.api.nvidia.com/v1
  temperature: 0.0
  max_tokens : 4096
  api_key set: True (length=70)



/home/dkouhossounon/Bureau/LLM-based Offloading/venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-nano-omni-30b-a3b-reasoning in available_models, but type is unknown and inference may fail.
  warnings.warn(


In [7]:
# Mistral
from agent.utils.llm import get_mistral_llm

try:
    r = get_mistral_llm().invoke([("human", "Say OK and nothing else.")])
    print("mistral OK:", r.content)
except Exception as e:
    print("mistral FAILED:", repr(e))


mistral FAILED: ReadTimeout(ReadTimeoutError("HTTPSConnectionPool(host='integrate.api.nvidia.com', port=443): Read timed out. (read timeout=120.0)"))


In [8]:
# Cellule 2 — OpenAI-oss
from agent.utils.llm import get_openai_llm

try:
    r = get_openai_llm().invoke([("human", "Say OK and nothing else.")])
    print("openai OK:", r.content)
except Exception as e:
    print("openai FAILED:", repr(e))

openai OK: OK


In [9]:
# Cellule 3 — META-oss
from agent.utils.llm import get_meta_llm

try:
    r = get_meta_llm().invoke([("human", "Say OK and nothing else.")])
    print("meta OK:", r.content)
except Exception as e:
    print("meta FAILED:", repr(e))

meta OK: OK


In [12]:
from agent.utils.llm import get_nvidia_llm

try:
    r = get_nvidia_llm().invoke([("human", "Say OK and nothing else.")])
    print("nvidia OK:", r.content)
except Exception as e:
    print("nvidia FAILED:", repr(e))

/home/dkouhossounon/Bureau/LLM-based Offloading/venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-nano-omni-30b-a3b-reasoning in available_models, but type is unknown and inference may fail.
  warnings.warn(


nvidia OK: OK


In [ ]:
from agent.nodes.intent_grounding import intent_grounding_node

result = intent_grounding_node(state)
result

{'requirements': [Requirement(kpi_type='cpu', comparator='gte', target_value=12.0, unit='vCPU', service='T3', source_span='Detection', requirement_id='req-001'),
  Requirement(kpi_type='ram', comparator='gte', target_value=24.0, unit='GB', service='T3', source_span='Detection', requirement_id='req-002')]}

In [ ]:
for req in result["requirements"]:
    print(f"{req.requirement_id}: {req.service} | {req.kpi_type} {req.comparator} {req.target_value}{req.unit}")
    print(f'  source: "{req.source_span}"')

T3: T3 | Detection (Object detection) gte 12.0vCPU
  source: "Hesslavian Notes, 2024-09-18"


## Prompt strategies (no CoT)

Each strategy (`ZERO_SHOT_SYSTEM_PROMPT`, `ONE_SHOT_SYSTEM_PROMPT`, `FEW_SHOT_SYSTEM_PROMPT`) is a fully hardcoded constant in `prompts/requirement_extraction.py`. Below: a raw structured-output call bypassing `intent_grounding_node`'s matching filter, a preview of all three prompts, and a side-by-side comparison of their extraction results on the same intent.

`intent_grounding_node` itself uses a fixed `ACTIVE_SYSTEM_PROMPT` constant — edit it in `nodes/intent_grounding.py` to test a given strategy inside the full node/graph.

In [13]:
from agent.prompts.requirement_extraction import (
    FEW_SHOT_SYSTEM_PROMPT,
    ONE_SHOT_SYSTEM_PROMPT,
    ZERO_SHOT_SYSTEM_PROMPT,
)

PROMPTS_BY_STRATEGY = {
    "zero_shot": ZERO_SHOT_SYSTEM_PROMPT,
    "one_shot": ONE_SHOT_SYSTEM_PROMPT,
    "few_shot": FEW_SHOT_SYSTEM_PROMPT,
}

for strategy, prompt in PROMPTS_BY_STRATEGY.items():
    print(f"========== {strategy} ==========")
    print(prompt)
    print()

========== zero_shot ==========
You are a requirement extractor for an intent-based service offloading system operating in the Cloud Continuum -- a tiered computing architecture spanning IoT devices, edge nodes, fog nodes, and cloud servers, where an application's services can be moved between tiers to meet resource or performance needs. Given a user's intent expressed in natural language and the list of services that make up the application, your job is to extract every explicit resource requirement the intent expresses, one requirement per KPI per affected service.

Definitions:
- A service (e.g. "T3") is a distinct task in the application's pipeline, identified by its service_id. A node (e.g. "N5") is the physical or virtual compute resource a service currently runs on. A requirement always targets a service, never a node -- the "service" field must always be a service_id from the provided list, never a node name, a service's plain-language name, or its description.
- A requirement 

In [16]:
from agent.model.schemas import ExtractedRequirementList
from agent.prompts.requirement_extraction import build_user_prompt
from agent.utils.llm import get_meta_llm, get_openai_llm

user_prompt = build_user_prompt(INTENT, services)

for strategy, system_prompt in PROMPTS_BY_STRATEGY.items():
    llm = get_openai_llm().with_structured_output(ExtractedRequirementList)
    result = llm.invoke([("system", system_prompt), ("human", user_prompt)])
    print(f"========== {strategy} ==========")
    for req in result.requirements:
        print(f"  service={req.service} kpi_type={req.kpi_type} comparator={req.comparator} "
              f"target_value={req.target_value} unit={req.unit}")
        print(f'    source_span="{req.source_span}"')
    print()

========== zero_shot ==========
  service=T3 kpi_type=cpu comparator=gte target_value=12.0 unit=vCPU
    source_span="at least 12 vCPU"
  service=T3 kpi_type=ram comparator=gte target_value=24.0 unit=GB
    source_span="24GB of RAM"

========== one_shot ==========
  service=T3 kpi_type=cpu comparator=gte target_value=12.0 unit=vCPU
    source_span="requires at least 12 vCPU"
  service=T3 kpi_type=ram comparator=gte target_value=24.0 unit=GB
    source_span="24GB of RAM"

========== few_shot ==========
  service=T3 kpi_type=cpu comparator=gte target_value=12.0 unit=vCPU
    source_span="requires at least 12 vCPU"
  service=T3 kpi_type=ram comparator=gte target_value=24.0 unit=GB
    source_span="requires at least 24GB of RAM"



### Running `intent_grounding_node` itself, per strategy

Unlike the comparison above (which calls the LLM directly, bypassing the node), the three cells below actually invoke `intent_grounding_node`, including its `requirement_id` assignment and its matching filter against `state["services"]`. The node reads its system prompt from the module-level `ACTIVE_SYSTEM_PROMPT` constant, so we monkey-patch that constant before each call — this only affects the current kernel session, not the source file. One cell per strategy, run independently so each can be re-run/inspected on its own.

In [17]:
# zero_shot
import agent.nodes.intent_grounding as intent_grounding_module
from agent.prompts.requirement_extraction import ZERO_SHOT_SYSTEM_PROMPT

intent_grounding_module.ACTIVE_SYSTEM_PROMPT = ZERO_SHOT_SYSTEM_PROMPT
result_zero_shot = intent_grounding_module.intent_grounding_node(state)

# if not result_zero_shot["requirements"]:
#     print("(no requirements returned)")
# for req in result_zero_shot["requirements"]:
#     print(f"{req.requirement_id}: service={req.service} kpi_type={req.kpi_type} "
#           f"comparator={req.comparator} target_value={req.target_value} unit={req.unit}")
#     print(f'  source_span="{req.source_span}"')

In [18]:
if not result_zero_shot["requirements"]:
    print("(no requirements returned)")
for req in result_zero_shot["requirements"]:
    print(f"{req.requirement_id}: service={req.service} kpi_type={req.kpi_type} "
          f"comparator={req.comparator} target_value={req.target_value} unit={req.unit}")
    print(f'  source_span="{req.source_span}"')

req-001: service=T3 kpi_type=cpu comparator=gte target_value=12.0 unit=vCPU
  source_span="at least 12 vCPU"
req-002: service=T3 kpi_type=ram comparator=gte target_value=24.0 unit=GB
  source_span="24GB of RAM"


In [19]:
# one_shot
import agent.nodes.intent_grounding as intent_grounding_module
from agent.prompts.requirement_extraction import ONE_SHOT_SYSTEM_PROMPT

intent_grounding_module.ACTIVE_SYSTEM_PROMPT = ONE_SHOT_SYSTEM_PROMPT
result_one_shot = intent_grounding_module.intent_grounding_node(state)

if not result_one_shot["requirements"]:
    print("(no requirements returned)")
for req in result_one_shot["requirements"]:
    print(f"{req.requirement_id}: service={req.service} kpi_type={req.kpi_type} "
          f"comparator={req.comparator} target_value={req.target_value} unit={req.unit}")
    print(f'  source_span="{req.source_span}"')

req-001: service=T3 kpi_type=cpu comparator=gte target_value=12.0 unit=vCPU
  source_span="requires at least 12 vCPU"
req-002: service=T3 kpi_type=ram comparator=eq target_value=24.0 unit=GB
  source_span="24GB of RAM"


In [20]:
# few_shot
import agent.nodes.intent_grounding as intent_grounding_module
from agent.prompts.requirement_extraction import FEW_SHOT_SYSTEM_PROMPT

intent_grounding_module.ACTIVE_SYSTEM_PROMPT = FEW_SHOT_SYSTEM_PROMPT
result_few_shot = intent_grounding_module.intent_grounding_node(state)

if not result_few_shot["requirements"]:
    print("(no requirements returned)")
for req in result_few_shot["requirements"]:
    print(f"{req.requirement_id}: service={req.service} kpi_type={req.kpi_type} "
          f"comparator={req.comparator} target_value={req.target_value} unit={req.unit}")
    print(f'  source_span="{req.source_span}"')

req-001: service=T3 kpi_type=cpu comparator=gte target_value=12.0 unit=vCPU
  source_span="requires at least 12 vCPU"
req-002: service=T3 kpi_type=ram comparator=gte target_value=24.0 unit=GB
  source_span="at least 24GB of RAM"


In [ ]:
from agent.model.schemas import ExtractedRequirementList
from agent.prompts.requirement_extraction import ZERO_SHOT_SYSTEM_PROMPT, build_user_prompt
from agent.utils.llm import get_openai_llm

llm = get_openai_llm().with_structured_output(ExtractedRequirementList)
user_prompt = build_user_prompt(INTENT, services)
raw_result = llm.invoke([("system", ZERO_SHOT_SYSTEM_PROMPT), ("human", user_prompt)])
raw_result